---
---
# EL SERVIDOR LOCAL DE IA · desde tu cuaderno
## La misma llamada, otra máquina

**Curso Big Data e IA Aplicada · Formación San Miguel · Edición Técnica**

En el LAB14 llamaste a Gemini con `urllib` y un JSON. Aquí vas a llamar a un modelo que corre en
una máquina de vuestra red — **y vas a ver que cambia muy poco**.

> 🎯 **La idea del cuaderno:** un `POST` con JSON de ida y un JSON de vuelta. **La forma de hablar
> con una máquina remota no depende de quién sea la máquina.**

## Paso 1 · ¿Está vivo el servidor?

Antes de pedirle nada, se comprueba que responde. Es el mismo hábito que con el cuadro de mando:
**que el servicio viva no significa que sirva**, pero si no vive, no hay nada que hacer.

In [ ]:
import json, urllib.request, urllib.error

SERVIDOR = "http://IP_DEL_SERVIDOR:11434"     # <-- cambialo por el tuyo

try:
    with urllib.request.urlopen(f"{SERVIDOR}/api/tags", timeout=10) as r:
        modelos = [m["name"] for m in json.load(r).get("models", [])]
    print(f"  El servidor responde. Modelos disponibles: {len(modelos)}")
    for m in modelos:
        print(f"    - {m}")
except urllib.error.URLError as e:
    print(f"  NO responde: {e.reason}")
    print("  Comprueba la IP, el puerto 11434 y el cortafuegos del servidor.")

## Paso 2 · La función · compárala con la del LAB14

**Ponlas una al lado de la otra.** Las diferencias son tres: la URL, el nombre del campo del texto
que vuelve, y que aquí no hace falta clave.

In [ ]:
import json, urllib.request, urllib.error

SERVIDOR = "http://IP_DEL_SERVIDOR:11434"
MODELO = "llama3.2:3b"        # el que tenga tu servidor: mira el paso 1

def ia_local(prompt, temperatura=0.2):
    """Misma forma que gemini() del LAB14: POST con JSON, y JSON de vuelta."""
    cuerpo = {"model": MODELO, "prompt": prompt, "stream": False,
              "options": {"temperature": temperatura}}
    peticion = urllib.request.Request(
        f"{SERVIDOR}/api/generate",
        data=json.dumps(cuerpo).encode("utf-8"),
        headers={"Content-Type": "application/json"}, method="POST")
    try:
        with urllib.request.urlopen(peticion, timeout=300) as r:
            return json.load(r)["response"]
    except urllib.error.HTTPError as e:
        return f"[HTTP {e.code}] revisa el nombre del modelo"
    except urllib.error.URLError as e:
        return f"[sin conexion] {e.reason}"

print(ia_local("Responde solo con el numero: cuanto es 2+2"))

### Las tres diferencias, en una tabla

| | Gemini (LAB14) | Tu servidor |
|---|---|---|
| URL | `generativelanguage.googleapis.com/…` | `http://IP:11434/api/generate` |
| Clave | `?key=API_KEY` | **No hace falta** |
| El texto vuelve en | `candidates[0].content.parts[0].text` | `response` |
| La temperatura | `generationConfig.temperature` | `options.temperature` |

**Todo lo demás es idéntico.** El `POST`, el `Content-Type`, el JSON y el `urllib`.

## Paso 3 · Por qué la segunda llamada va más rápida

La primera **carga el modelo en memoria**. La segunda ya lo encuentra cargado.

In [ ]:
import time

t = time.time()
ia_local("Di solo: hola")
primera = time.time() - t

t = time.time()
ia_local("Di solo: adios")
segunda = time.time() - t

print(f"  primera llamada: {primera:5.1f} s")
print(f"  segunda llamada: {segunda:5.1f} s")
if segunda > 0:
    print(f"  la segunda es {primera/segunda:.1f} veces mas rapida")
print()
print("  La diferencia es la CARGA del modelo en memoria. Con OLLAMA_KEEP_ALIVE")
print("  configurado, se queda cargado y esa espera solo se paga una vez.")

## Paso 4 · La velocidad real, en tokens por segundo

La respuesta de Ollama **trae los tiempos dentro**. No hay que cronometrar: viene medido.

In [ ]:
import json, urllib.request

SERVIDOR = "http://IP_DEL_SERVIDOR:11434"
MODELO = "llama3.2:3b"

cuerpo = {"model": MODELO, "stream": False,
          "prompt": "Explica en un parrafo que es un ancla de datos."}
peticion = urllib.request.Request(
    f"{SERVIDOR}/api/generate",
    data=json.dumps(cuerpo).encode("utf-8"),
    headers={"Content-Type": "application/json"}, method="POST")

with urllib.request.urlopen(peticion, timeout=300) as r:
    d = json.load(r)

tokens = d.get("eval_count", 0)
nanos = d.get("eval_duration", 1)
print(f"  tokens generados : {tokens}")
print(f"  velocidad        : {tokens / (nanos / 1e9):.1f} tokens/s")
print(f"  carga del modelo : {d.get('load_duration', 0) / 1e9:.2f} s")
print(f"  total            : {d.get('total_duration', 0) / 1e9:.2f} s")
print()
print(d["response"][:300])

| Velocidad | Sensación |
|---|---|
| Más de 20 tokens/s | Cómodo: se lee mientras escribe |
| 5–20 | Se nota, pero se usa |
| Menos de 5 | Para procesos por lotes, no para conversar |

## Paso 5 · ⚓ Y ahora lo que de verdad importa: audítalo

**Un modelo local se equivoca igual que uno de pago.** A veces más. El protocolo no cambia:
pídele algo que puedas contrastar contra un número tuyo.

In [ ]:
ANCLA = 999535

pregunta = ("Tengo una tabla de ventas. Si el fichero original tiene 1000000 "
            "de filas y quito 465 con precio negativo, cuantas quedan? "
            "Responde solo con el numero.")

respuesta = ia_local(pregunta, temperatura=0.0)
print(f"  Su respuesta : {respuesta.strip()[:80]}")
print(f"  Tu ancla     : {ANCLA}")
print()
print("  Coincide? Y si coincide: lo ha CALCULADO, o estaba en el enunciado?")
print("  Preguntale algo cuyo resultado NO se pueda leer de la pregunta.")

> 🎯 **Que acierte no demuestra que sepa.** Aquí la resta está en el enunciado: lo raro sería que
> fallara. Cámbiale la pregunta por una donde el resultado no se pueda leer del texto, y verás la
> diferencia entre un modelo de 3B y uno de pago.
>
> **Y esa diferencia es el precio honesto de que el dato no salga de tu máquina.**

## Lo que te llevas

| |
|---|
| La llamada es **la misma**: `POST` con JSON, JSON de vuelta |
| Cambian tres cosas: la URL, el campo del texto y que **no hace falta clave** |
| La primera llamada **carga el modelo**; la segunda ya lo encuentra |
| La respuesta trae **los tiempos medidos dentro**: `eval_count` y `eval_duration` |
| ⚓ Un modelo local **se audita igual**: contra un ancla tuya |
| !Y responde peor. **Ese es el precio de que el dato no salga** |

---

*El servidor local de IA desde tu cuaderno · Formación San Miguel · Zaragoza*